# 📱 Used Phone Resale Price Predictor — ANN Training (Google Colab)

This notebook trains a feed-forward ANN (Keras) on the used-phone resale price dataset,
evaluates it, and saves the trained model + preprocessors so they can be downloaded
and used later in the Streamlit app.

**Before running:** Runtime → Change runtime type → set Hardware accelerator to **GPU** (optional, speeds up training).

## 1. Install / import libraries
Colab already ships with pandas, numpy, scikit-learn and TensorFlow, so this just imports them.

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2. Get the dataset into Colab
Pick **one** of the two options below.

- **Option A — Google Drive (recommended)**: if `used_phone_price_prediction_1M.csv` is already in your Drive, this mounts your Drive so the file can be read directly. Update `DATA_PATH` to match where the file lives.
- **Option B — Direct upload**: uploads the CSV straight from your computer for this session (slower for a 1M-row file, and you'll need to re-upload every time the runtime resets).

In [ ]:
# --- Option A: Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/used_phone_price_prediction_1M.csv"  # <-- update this path

In [ ]:
# --- Option B: Direct upload (skip this cell if you used Option A) ---
# from google.colab import files
# uploaded = files.upload()  # select used_phone_price_prediction_1M.csv from your computer
# DATA_PATH = "used_phone_price_prediction_1M.csv"

## 3. Load the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Data loaded. Shape:", df.shape)
df.head()

## 4. Drop columns that don't help much
These were checked earlier using correlation and feature-importance analysis and barely affect `resale_price`.

In [ ]:
columns_to_drop = [
    "screen_size_inches",
    "usage_hours_per_day",
    "city_tier",
    "seller_type",
    "charger_available",
    "release_year",
    "processor_score",
    "camera_score"
]

df = df.drop(columns=columns_to_drop)
print("Columns after dropping:", df.shape[1])

## 5. Separate input (X) and target (y)

In [ ]:
X = df.drop("resale_price", axis=1)
y = df["resale_price"]

## 6. One-hot encode text columns
ANN can only understand numbers, so `brand`/`model`/`condition`/`os_type` are converted into 0/1 columns.

In [ ]:
categorical_cols = ["brand", "model", "condition", "os_type"]
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Save the column names now - the app needs the EXACT same
# columns later when it makes a single prediction.
feature_names = X_encoded.columns.tolist()
print("Total features after encoding:", len(feature_names))

## 7. Split into train / validation / test

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_encoded, y, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train size:", X_train.shape)
print("Val size:  ", X_val.shape)
print("Test size: ", X_test.shape)

## 8. Scale the data
Neural networks train better when numbers are on a similar scale (mean 0, std 1).

In [ ]:
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_val_scaled = x_scaler.transform(X_val)
X_test_scaled = x_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_val_scaled = y_scaler.transform(y_val.values.reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1)).ravel()

## 9. Build the ANN model
3 hidden layers (128 → 64 → 32) with BatchNorm + Dropout for regularization.

In [ ]:
n_features = X_train_scaled.shape[1]

model = Sequential([
    Dense(128, activation="relu", input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.2),

    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),

    Dense(32, activation="relu"),
    Dropout(0.1),

    Dense(1, activation="linear")
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()

## 10. Callbacks
`EarlyStopping` stops training once the model stops improving. `ReduceLROnPlateau` slows the learning rate down if progress stalls.

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
]

## 11. Train the model
On Colab's free GPU this should be noticeably faster than CPU-only. Adjust `epochs`/`batch_size` if needed.

In [ ]:
history = model.fit(
    X_train_scaled, y_train_scaled,
    validation_data=(X_val_scaled, y_val_scaled),
    epochs=100,
    batch_size=256,
    callbacks=callbacks,
    verbose=2
)

## 12. Plot training curves (optional, useful for the README)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss (scaled)")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

## 13. Evaluate on the test set

In [ ]:
predictions_scaled = model.predict(X_test_scaled).ravel()

# convert predictions back to real rupees
predictions = y_scaler.inverse_transform(predictions_scaled.reshape(-1, 1)).ravel()

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("--- Final Test Results ---")
print("MAE (average error in rupees):", round(mae, 2))
print("R2 score (0 to 1, higher is better):", round(r2, 4))

## 14. Save the model + preprocessing tools
Saves into a local `models/` folder inside the Colab runtime, then copies it to your
mounted Google Drive so it survives after the runtime disconnects (if you used Option A above).
If you used Option B (direct upload), it instead offers a zip download.

In [ ]:
import os
os.makedirs("models", exist_ok=True)

model.save("models/phone_price_ann_model.keras")
joblib.dump(x_scaler, "models/x_scaler.pkl")
joblib.dump(y_scaler, "models/y_scaler.pkl")
joblib.dump(feature_names, "models/feature_names.pkl")

print("Model and files saved successfully in the 'models' folder.")

In [ ]:
# --- If you mounted Google Drive (Option A): copy the models/ folder to Drive ---
import shutil

DRIVE_SAVE_DIR = "/content/drive/MyDrive/phone-price-predictor/models"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

for fname in os.listdir("models"):
    shutil.copy(f"models/{fname}", f"{DRIVE_SAVE_DIR}/{fname}")

print(f"Copied trained model files to {DRIVE_SAVE_DIR}")

In [ ]:
# --- If you used direct upload (Option B) instead: download the models/ folder as a zip ---
# import shutil
# from google.colab import files
#
# shutil.make_archive("models", "zip", "models")
# files.download("models.zip")